In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "datasets").exists():
            return candidate
    raise RuntimeError("Repository root not found.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
# Dataset Generation Pipeline (multi-call by slot combinations)
# This notebook reads your prompt config (intent_prompts_full_v2.json) and executes
# multiple API calls per intent, one per slot-combination "mode", until it fills the target counts.
#
# Requirements:
#   pip install openai python-dotenv
#
# Files expected in the same folder as this notebook:
#   - intent_prompts_full_v2.json   (generated earlier)
#
# Output:
#   - one JSON dataset file per intent inside OUTPUT_DIR

import json
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple, Set, Any

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# =============================
# CONFIG: edit only here
# =============================
PROMPTS_PATH = REPO_ROOT / "datasets/field_operations/metadata/prompts/intent_prompts_full_v2.json"   # <-- your updated prompt file
OUTPUT_DIR = REPO_ROOT / "datasets/field_operations/raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose intents:
# - set to ["Confirm Movement Milestone"] to test a single intent
# - set to "ALL" to generate for every intent in the prompts file
INTENTS_TO_GENERATE = ["Lookup Operational Record"]  # or "ALL"

# Model params
MODEL = "gpt-4.1"
TEMPERATURE_INITIAL = 0.25
TEMPERATURE_DELTA = 0.35
MAX_OUTPUT_TOKENS = 32768

# Validation params (should match your prompt rules)
MIN_WORDS = 5
MAX_WORDS = 25
MAX_ENTITY_SPAN_WORDS = 4

# Batch size per API call (smaller reduces duplicates)
BATCH_SIZE = 40

# =============================
# Load prompt config
# =============================
PROMPT_CFG: Dict[str, Any] = json.loads(PROMPTS_PATH.read_text(encoding="utf-8"))

# If you ever want global instructions, put them here.
INSTRUCTIONS = (
    "You are generating an English slot-filling dataset.\n"
    "Output MUST be valid JSON that strictly follows the provided JSON Schema.\n"
    "Do not include any extra keys, comments, markdown, or explanations.\n"
)

def apply_tokens(template: str, replacements: Dict[str, str]) -> str:
    """Simple token replacement. We avoid .format() because prompts may contain '{ }' JSON snippets."""
    out = template
    for k, v in replacements.items():
        out = out.replace(k, v)
    return out

def full_utterance(example: dict) -> str:
    return "".join(seg["text"] for seg in example["data"]).strip()

def normalize_utt(text: str) -> str:
    return " ".join(text.split()).strip().lower()

def entities_counts(example: dict) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for seg in example["data"]:
        if "entity" in seg:
            counts[seg["entity"]] = counts.get(seg["entity"], 0) + 1
    return counts

def has_plain_segment(example: dict) -> bool:
    return any("entity" not in seg for seg in example["data"])

# Segmentation range map requested:
# K = number of tagged entities required in a mode.
SEG_RANGE_BY_K = {
    1: (2, 5),
    2: (3, 7),
    3: (7, 9),
    4: (8, 11),
}


In [45]:
# Helper: build schema + validate + call model

def build_segment_schema(slot_labels: List[str]) -> Dict[str, Any]:
    """Segment can be plain {text} or entity {text, entity}."""
    plain = {
        "type": "object",
        "additionalProperties": False,
        "properties": {"text": {"type": "string", "minLength": 1, "maxLength": 140}},
        "required": ["text"],
    }
    entity = {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "text": {"type": "string", "minLength": 1, "maxLength": 30},
            "entity": {"type": "string", "enum": slot_labels},
        },
        "required": ["text", "entity"],
    }
    return {"anyOf": [plain, entity]}

def build_schema_for_mode(intent_name: str, slot_labels: List[str], n: int, k_required: int) -> Dict[str, Any]:
    """Build strict JSON schema for EXACTLY n examples, with 'data' length range based on K."""
    seg_schema = build_segment_schema(slot_labels)
    min_seg, max_seg = SEG_RANGE_BY_K.get(k_required, (2*k_required + 1, 2*k_required + 3))
    # fallback is conservative, but your current intents use K<=3 mostly.

    data_schema = {
        "type": "array",
        "minItems": min_seg,
        "maxItems": max_seg,
        "items": seg_schema,
    }

    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            intent_name: {
                "type": "array",
                "minItems": n,
                "maxItems": n,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {"data": data_schema},
                    "required": ["data"],
                },
            }
        },
        "required": [intent_name],
    }

def is_valid_example_for_mode(
    example: dict,
    slot_labels: List[str],
    required: List[str],
    forbidden: List[str],
    min_words: int = MIN_WORDS,
    max_words: int = MAX_WORDS,
) -> Tuple[bool, str]:
    """Validate semantics that schema can't express: counts, required/forbidden, span size, etc."""
    utt = full_utterance(example)
    wc = len(utt.split())
    if wc < min_words or wc > max_words:
        return False, f"length={wc}"

    if not has_plain_segment(example):
        return False, "no_plain_segment"

    counts = entities_counts(example)

    # Each slot label at most once (no multi-tags for same label)
    for s in slot_labels:
        if counts.get(s, 0) > 1:
            return False, f"too_many_{s}"

    # Required must appear exactly once
    for s in required:
        if counts.get(s, 0) != 1:
            return False, f"missing_required_{s}"

    # Forbidden must not appear
    for s in forbidden:
        if counts.get(s, 0) != 0:
            return False, f"forbidden_present_{s}"

    # Entity spans must be short (1–4 words)
    for seg in example["data"]:
        if "entity" in seg and len(seg["text"].split()) > MAX_ENTITY_SPAN_WORDS:
            return False, f"entity_span_too_long_{seg['entity']}"

    return True, "ok"

def build_prompt_for_mode(
    intent_name: str,
    template: str,
    n: int,
    required: List[str],
    forbidden: List[str],
    blacklist: List[str],
) -> str:
    return apply_tokens(
        template,
        {
            "__N__": str(n),
            "__INTENT__": intent_name,
            "__REQUIRED_ENTITIES_JSON__": json.dumps(required, ensure_ascii=False),
            "__FORBIDDEN_ENTITIES_JSON__": json.dumps(forbidden, ensure_ascii=False),
            "__BLACKLIST_JSON__": json.dumps(blacklist, ensure_ascii=False),
        },
    )

def request_batch(
    intent_name: str,
    slot_labels: List[str],
    prompt_template: str,
    n: int,
    required: List[str],
    forbidden: List[str],
    blacklist: List[str],
    temperature: float,
) -> List[dict]:
    schema = build_schema_for_mode(intent_name, slot_labels, n=n, k_required=len(required))
    user_prompt = build_prompt_for_mode(
        intent_name=intent_name,
        template=prompt_template,
        n=n,
        required=required,
        forbidden=forbidden,
        blacklist=blacklist,
    )

    resp = client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=user_prompt,
        temperature=temperature,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        text={
            "format": {
                "type": "json_schema",
                "name": f"{intent_name.lower().replace(' ', '_')}_mode_batch",
                "strict": True,
                "schema": schema,
            }
        },
    )

    if getattr(resp, "status", None) == "incomplete":
        raise RuntimeError(f"Incomplete response: {resp.incomplete_details}")

    obj = json.loads(resp.output_text)
    return obj[intent_name]


In [46]:
# Main: generate one intent by iterating its slot-combination modes (multiple API calls)

def generate_intent_dataset(intent_name: str) -> Dict[str, List[dict]]:
    intent_cfg = PROMPT_CFG[intent_name]
    slot_labels: List[str] = intent_cfg["slot_labels"]
    combinations: List[Dict[str, Any]] = intent_cfg.get("combinations", [])
    prompt_template = intent_cfg["delta"]  # delta supports blacklist; we use it for ALL calls

    accepted: List[dict] = []
    accepted_norm: Set[str] = set()

    print(f"\n=== INTENT: {intent_name} ===")
    print(f"Slots: {slot_labels}")
    if intent_cfg.get("notes"):
        print(f"Notes: {intent_cfg['notes']}")
    print(f"Modes: {len(combinations)}")

    for mode in combinations:
        mode_name = mode["name"]
        required = mode.get("required", [])
        forbidden = mode.get("forbidden", [])
        target = int(mode.get("target", 0))

        print(f"\n-- Mode: {mode_name} | target={target} | required={required} | forbidden={forbidden}")

        mode_accepted = 0
        attempt = 0
        temperature = TEMPERATURE_INITIAL

        while mode_accepted < target:
            attempt += 1
            need = target - mode_accepted
            n_req = min(need, BATCH_SIZE)

            # Global blacklist (avoid repeating across ALL modes)
            blacklist = [full_utterance(ex) for ex in accepted]

            batch = request_batch(
                intent_name=intent_name,
                slot_labels=slot_labels,
                prompt_template=prompt_template,
                n=n_req,
                required=required,
                forbidden=forbidden,
                blacklist=blacklist,
                temperature=temperature,
            )

            added = 0
            for ex in batch:
                utt = " ".join(full_utterance(ex).split())
                key = normalize_utt(utt)

                if key in accepted_norm:
                    continue

                ok, reason = is_valid_example_for_mode(
                    example=ex,
                    slot_labels=slot_labels,
                    required=required,
                    forbidden=forbidden,
                )
                if not ok:
                    continue

                accepted.append(ex)
                accepted_norm.add(key)
                mode_accepted += 1
                added += 1

                if mode_accepted >= target:
                    break

            print(f"  attempt={attempt} requested={n_req} added={added} mode_progress={mode_accepted}/{target} total={len(accepted)}")

            # If we got stuck, increase temperature slightly to escape repetition
            if added == 0:
                temperature = min(0.7, temperature + 0.1)

            # Safety guard to avoid endless loops in notebook runs:
            if attempt > 50 and mode_accepted < target:
                raise RuntimeError(f"Too many attempts for mode '{mode_name}'. Consider increasing temperature or relaxing constraints.")

    # Save per-intent file
    out_path = OUTPUT_DIR / f"{intent_name.lower().replace(' ', '_')}_300.json"
    out_obj = {intent_name: accepted}
    out_path.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nSaved: {out_path} (count={len(accepted)})")
    return out_obj


In [47]:
# RUN: generate datasets (single intent or ALL)

intents = list(PROMPT_CFG.keys()) if INTENTS_TO_GENERATE == "ALL" else INTENTS_TO_GENERATE

for intent_name in intents:
    if intent_name not in PROMPT_CFG:
        raise KeyError(f"Intent '{intent_name}' not found in {PROMPTS_PATH}")
    generate_intent_dataset(intent_name)



=== INTENT: Incident Status ===
Slots: ['status', 'location']
Notes: ALL examples must include BOTH status and location exactly once. status MUST be a single past-tense status word. location MUST be 1–2 words max.
Modes: 1

-- Mode: both_only | target=300 | required=['status', 'location'] | forbidden=[]
  attempt=1 requested=40 added=23 mode_progress=23/300 total=23
  attempt=2 requested=40 added=40 mode_progress=63/300 total=63
  attempt=3 requested=40 added=39 mode_progress=102/300 total=102
  attempt=4 requested=40 added=33 mode_progress=135/300 total=135
  attempt=5 requested=40 added=40 mode_progress=175/300 total=175
  attempt=6 requested=40 added=39 mode_progress=214/300 total=214
  attempt=7 requested=40 added=15 mode_progress=229/300 total=229
  attempt=8 requested=40 added=40 mode_progress=269/300 total=269
  attempt=9 requested=31 added=30 mode_progress=299/300 total=299
  attempt=10 requested=1 added=1 mode_progress=300/300 total=300

Saved: generated_datasets\incident_sta

In [48]:
# Optional: check duplicates inside a saved dataset file (prints sentence + indices)

import re
from collections import defaultdict

def list_duplicates(dataset_path: Path, intent_name: str, ignore_punct: bool = False, max_groups: int = 50):
    obj = json.loads(dataset_path.read_text(encoding="utf-8"))
    examples = obj[intent_name]

    def norm2(s: str) -> str:
        s = " ".join(s.split()).strip().lower()
        if ignore_punct:
            s = re.sub(r"[^\w\s]", "", s)
            s = " ".join(s.split()).strip()
        return s

    m = defaultdict(list)
    original = {}
    for i, ex in enumerate(examples):
        utt = full_utterance(ex)
        k = norm2(utt)
        m[k].append(i)
        original.setdefault(k, utt)

    dups = [(k, idxs) for k, idxs in m.items() if len(idxs) > 1]
    dups.sort(key=lambda kv: (-len(kv[1]), min(kv[1])))

    print(f"Total examples: {len(examples)}")
    print(f"Unique (normalized): {len(m)}")
    print(f"Duplicate groups: {len(dups)}\n")

    for g, (k, idxs) in enumerate(dups[:max_groups], start=1):
        print("=" * 80)
        print(f"[Group {g}] Indices: {idxs}")
        print(f"Sentence: {original[k]}")

# Example:
out_file = OUTPUT_DIR / "confirm_movement_milestone_300.json"
list_duplicates(out_file, "Confirm Movement Milestone", ignore_punct=False)


Total examples: 300
Unique (normalized): 300
Duplicate groups: 0

